# Hillstrom Email Marketing — Dataset Collection
### Project: Promotional Email Campaign A/B Testing
**Objective:** Collect historical randomized email-campaign data and verify that it is ready for A/B testing.  
**Source:** [scikit-uplift: fetch_hillstrom](https://www.uplift-modeling.com/en/latest/api/datasets/fetch_hillstrom.html) (Kevin Hillstrom / MineThatData).  
**Note:** These are historical experiment records, not live promotional emails. We will analyze whether promotional emails increased customer conversion, visits, and revenue in later notebook sections.

## Step 1 — Install the dataset library
**Why?** `scikit-uplift` provides a reproducible downloader for the public Hillstrom dataset. Run once per new Colab runtime.

In [1]:
%pip -q install scikit-uplift

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Import libraries
**Why?** `pandas` lets us inspect, organize and export customer records.

In [2]:
import pandas as pd
from pathlib import Path
from sklift.datasets import fetch_hillstrom

pd.set_option("display.max_columns", None)

c:\Users\Sakthi Jayakumar\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3 — Download the dataset
**Why?** We need the customer attributes, campaign assignment, and *all three* outcomes (`visit`, `conversion`, `spend`). `target_col="all"` requests all outcomes.

In [3]:
dataset = fetch_hillstrom(target_col="all")
print("Dataset downloaded successfully")
print("Feature shape:", dataset.data.shape)
print("Outcome shape:", dataset.target.shape)

Hillstrom dataset: 100%|██████████| 443k/443k [00:01<00:00, 241kiB/s]


Dataset downloaded successfully
Feature shape: (64000, 8)
Outcome shape: (64000, 3)


## Step 4 — Combine features, campaign groups, and outcomes
**Why?** The loader separates customer features, treatment assignments, and outcomes. One DataFrame is easier to analyze and export.

In [4]:
df = dataset.data.copy()
df["segment"] = dataset.treatment.to_numpy()
for col in dataset.target.columns:
    df[col] = dataset.target[col].to_numpy()

print("Complete dataset shape:", df.shape)
display(df.head(10))

Complete dataset shape: (64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0
5,6,2) $100 - $200,134.83,0,1,Surburban,0,Phone,Womens E-Mail,1,0,0.0
6,9,3) $200 - $350,280.20,1,0,Surburban,1,Phone,Womens E-Mail,0,0,0.0
7,9,1) $0 - $100,46.42,0,1,Urban,0,Phone,Womens E-Mail,0,0,0.0
8,9,5) $500 - $750,675.07,1,1,Rural,1,Phone,Mens E-Mail,0,0,0.0
9,10,1) $0 - $100,32.84,0,1,Urban,1,Web,Womens E-Mail,0,0,0.0


## Step 5 — Understand the experiment groups
- **No E-Mail** — control (received no promotional email)
- **Mens E-Mail** — treatment A
- **Womens E-Mail** — treatment B

**Why?** Never combine groups before identifying which users received each campaign.

In [ ]:
print("Campaign groups and customer counts:")
display(df["segment"].value_counts(dropna=False).rename_axis("campaign_group").reset_index(name="customers"))
print("Columns:", df.columns.tolist())

## Step 6 — Check collection quality (without altering original data)
**Why?** Check missing values, duplicates, data types and valid outcomes before deciding on preprocessing. Duplicate rows are not necessarily duplicate customers when no unique customer ID is provided: do not drop them automatically.

In [ ]:
print("Rows:", len(df))
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))
print("Fully identical rows:", int(df.duplicated().sum()))
print("Data types:")
display(df.dtypes.astype(str).to_frame("dtype"))
print("Outcome values:")
for column in ["visit", "conversion"]:
    print(column, df[column].value_counts(dropna=False).to_dict())
print("Spend summary:")
display(df["spend"].describe())

## Step 7 — Validate the collected dataset
**Why?** Stop early if downloading or merging silently creates a wrong dataset. The public Hillstrom release should have 64,000 customer records.

In [ ]:
assert len(df) == 64_000, f"Expected 64,000 rows, got {len(df)}"
assert {"segment", "visit", "conversion", "spend"}.issubset(df.columns)
assert df["segment"].nunique() == 3, "Expected 3 campaign groups"
assert df["visit"].dropna().isin([0, 1]).all()
assert df["conversion"].dropna().isin([0, 1]).all()
assert (df["spend"].dropna() >= 0).all()
print("Validation passed. Dataset ready for exploratory analysis and A/B testing.")

## Step 8 — Export collected data to CSV
**Why?** Preserve a reusable local copy of the collected dataset in your Colab runtime. Colab's `/content` storage is temporary; download the file or save it to Drive if you need it later.

In [ ]:
output_path = Path("/content/hillstrom_email_marketing.csv")
df.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Size (MB):", round(output_path.stat().st_size / 1_000_000, 2))

## Step 9 — Optional: Download the CSV to your computer
Run only if you want the file on your device.

In [ ]:
from google.colab import files
files.download(str(output_path))

## What comes next?
1. Exploratory data analysis (conversion, visit, and spending distribution)
2. Set hypotheses: **H₀:** email and control conversion rates are equal; **H₁:** they differ
3. Compare each treatment to control with appropriate statistical tests; consider multiple comparisons
4. Report effect sizes, confidence intervals, statistical significance, and business impact

**Important:** A historical experimental dataset cannot tell us whether real emails were delivered to your customers today. It can be used to learn and demonstrate campaign effectiveness analysis.